## TF-IDF (Term Frequency-Inverse Document Frequency)

## 1. Definition
**TF-IDF** is a statistical measure used to evaluate the importance of a word to a document in a collection or corpus. It increases proportionally to the number of times a word appears in the document but is offset by the frequency of the word in the corpus. This helps to adjust for the fact that some words appear more frequently in general (like "the", "is", "and") and are less important.

## 2. Syntax / Formula
The TF-IDF score is the product of two terms: **Term Frequency (TF)** and **Inverse Document Frequency (IDF)**.

$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

Where:
*   **$t$**: The specific term (word).
*   **$d$**: The specific document.
*   **$D$**: The total collection of documents (corpus).

### Term Frequency (TF)
Measures how frequently a term occurs in a document.

$$TF(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total number of words in } d}$$

### Inverse Document Frequency (IDF)
Measures how important a term is. While computing TF, all terms are considered equally important. However, certain terms, like "is", "of", and "that", may appear a lot but have little importance.

$$IDF(t, D) = \log\left(\frac{\text{Total number of documents } (N)}{\text{Number of documents containing term } t}\right)$$

## 3. Use Cases
*   **Information Retrieval**: Ranking search engine results by relevance.
*   **Keyword Extraction**: Identifying the most significant words in a document.
*   **Text Summarization**: Selecting sentences with high-weight words to form a summary.
*   **Document Clustering**: Grouping similar documents based on shared important keywords.
*   **Stop Word Filtering**: Automatically down-weighting common words without a manual stop word list.

## 4. Advantages vs. Disadvantages

| Feature | Advantages | Disadvantages |
| :--- | :--- | :--- |
| **Simplicity** | Easy to understand and implement; computationally efficient. | Does not capture **semantic meaning** (e.g., "happy" vs "joyful"). |
| **Relevance** | Effectively highlights words that are unique or specific to a document, filtering out common words. | Ignores **word order** and context (it is a "Bag of Words" technique). |
| **Baseline** | Serves as a strong baseline model for many NLP tasks before trying complex deep learning models. | Suffers from **high dimensionality** (vocabulary size) and data sparsity with large corpora. |
| **No Training** | Does not require training on a labeled dataset (unsupervised). | Cannot handle **Out of Vocabulary (OOV)** words well if the vocabulary is fixed. |

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
paragraph = """The history of computers is a remarkable saga of human ingenuity, evolving from primitive calculation aids to the powerful digital ecosystems that define modern existence. The journey began thousands of years ago with the abacus, a manual device used by ancient civilizations in Mesopotamia and China for basic arithmetic. However, the true theoretical ancestry of the computer traces back to the 19th century with Charles Babbage, a British mathematician who conceptualized the Analytical Engine. Although never completed in his lifetime, this mechanical design featured a central processing unit and memory, anticipating the architecture of modern machines. Ada Lovelace, his contemporary, famously wrote the first algorithm intended for this machine, earning her the title of the first computer programmer.

The electronic age dawned in the mid-20th century, driven by the urgencies of World War II. The ENIAC (Electronic Numerical Integrator and Computer), completed in 1945, stands as the first general-purpose electronic digital computer. It was a colossus, weighing 30 tons and utilizing 18,000 vacuum tubes, yet it could calculate artillery firing tables in seconds. The invention of the transistor at Bell Labs in 1947 marked a pivotal shift, replacing bulky and hot vacuum tubes with smaller, more efficient solid-state components. This ushered in the second generation of computers, which were faster, more reliable, and more energy-efficient.

The evolution accelerated in the late 1950s and 1960s with the development of the integrated circuit (IC) by Jack Kilby and Robert Noyce. The IC compressed thousands of transistors onto a single silicon chip, drastically reducing size and cost while boosting performance. This breakthrough led to the microprocessor in 1971, the Intel 4004, which placed the entire central processing unit on a single chip. This was the catalyst for the personal computer (PC) revolution of the 1970s and 80s, democratizing computing power through machines like the Apple II and the IBM PC.

The 1990s introduced the World Wide Web, transforming computers from isolated processing units into gateways for a global information network. This era of connectivity fundamentally altered global communication, commerce, and culture. As we moved into the 21st century, the paradigm shifted again towards mobility and ubiquity. Smartphones put the power of supercomputers into our pockets, while cloud computing decoupled processing power from physical hardware. Today, we stand on the frontier of a new era defined by artificial intelligence and quantum computing, technologies that promise to solve problems previously thought impossible, continuing the relentless trajectory of computational evolution."""

In [3]:
## we can also use sent_tokenize() to break the paragraph into sentences

sentences = nltk.sent_tokenize(paragraph.lower())
print(sentences)

['the history of computers is a remarkable saga of human ingenuity, evolving from primitive calculation aids to the powerful digital ecosystems that define modern existence.', 'the journey began thousands of years ago with the abacus, a manual device used by ancient civilizations in mesopotamia and china for basic arithmetic.', 'however, the true theoretical ancestry of the computer traces back to the 19th century with charles babbage, a british mathematician who conceptualized the analytical engine.', 'although never completed in his lifetime, this mechanical design featured a central processing unit and memory, anticipating the architecture of modern machines.', 'ada lovelace, his contemporary, famously wrote the first algorithm intended for this machine, earning her the title of the first computer programmer.', 'the electronic age dawned in the mid-20th century, driven by the urgencies of world war ii.', 'the eniac (electronic numerical integrator and computer), completed in 1945, s

In [4]:
## to tokenize the words from sentences, we can use word_tokenize
tokens = nltk.word_tokenize(paragraph.lower())
print(tokens)

['the', 'history', 'of', 'computers', 'is', 'a', 'remarkable', 'saga', 'of', 'human', 'ingenuity', ',', 'evolving', 'from', 'primitive', 'calculation', 'aids', 'to', 'the', 'powerful', 'digital', 'ecosystems', 'that', 'define', 'modern', 'existence', '.', 'the', 'journey', 'began', 'thousands', 'of', 'years', 'ago', 'with', 'the', 'abacus', ',', 'a', 'manual', 'device', 'used', 'by', 'ancient', 'civilizations', 'in', 'mesopotamia', 'and', 'china', 'for', 'basic', 'arithmetic', '.', 'however', ',', 'the', 'true', 'theoretical', 'ancestry', 'of', 'the', 'computer', 'traces', 'back', 'to', 'the', '19th', 'century', 'with', 'charles', 'babbage', ',', 'a', 'british', 'mathematician', 'who', 'conceptualized', 'the', 'analytical', 'engine', '.', 'although', 'never', 'completed', 'in', 'his', 'lifetime', ',', 'this', 'mechanical', 'design', 'featured', 'a', 'central', 'processing', 'unit', 'and', 'memory', ',', 'anticipating', 'the', 'architecture', 'of', 'modern', 'machines', '.', 'ada', 'lov

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

vectorizer = CountVectorizer(lowercase=True, stop_words='english', max_features=1500, dtype=np.int8)
vector = vectorizer.fit_transform(sentences).toarray()
vector[0]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8)

In [6]:
vectorizer.vocabulary_

{'history': np.int64(101),
 'computers': np.int64(59),
 'remarkable': np.int64(165),
 'saga': np.int64(169),
 'human': np.int64(103),
 'ingenuity': np.int64(109),
 'evolving': np.int64(88),
 'primitive': np.int64(155),
 'calculation': np.int64(41),
 'aids': np.int64(20),
 'powerful': np.int64(153),
 'digital': np.int64(75),
 'ecosystems': np.int64(79),
 'define': np.int64(69),
 'modern': np.int64(138),
 'existence': np.int64(89),
 'journey': np.int64(119),
 'began': np.int64(34),
 'thousands': np.int64(189),
 'years': np.int64(214),
 'ago': np.int64(19),
 'abacus': np.int64(15),
 'manual': np.int64(129),
 'device': np.int64(74),
 'used': np.int64(204),
 'ancient': np.int64(25),
 'civilizations': np.int64(49),
 'mesopotamia': np.int64(134),
 'china': np.int64(46),
 'basic': np.int64(33),
 'arithmetic': np.int64(29),
 'true': np.int64(198),
 'theoretical': np.int64(187),
 'ancestry': np.int64(24),
 'computer': np.int64(58),
 'traces': np.int64(193),
 '19th': np.int64(9),
 'century': np.i